# Acquire traces with Python

This is a code-based version of the acquisition example.
Please edit the settings below and run cells in order.
Choose a ChipWhisperer Lite/Husky, a supported VISA oscilloscope, or a PicoScope 3000E, then connect a target.
The default example uses a CW305 Shell AES RTL target with a VISA oscilloscope.
Configure the FPGA and wire the power/trigger signals before connecting; scope and target are separate devices.
See [setup](../docs/setup.md) and [hardware](../docs/hardware.md).

For interactive configuration, use the GUI examples for [ChipWhisperer](acquire_traces_cw_scope.ipynb) or [VISA](acquire_traces_with_visa_scope.ipynb).

In [1]:
from pathlib import Path

import chipwhisperer as cw
from cw_plugins.scopes import Oscilloscope, TriggerMode
from cw_plugins.scopes.base import ScopeBase
from cw_plugins.targets import CW305ShellExampleAES128BitRTL
from tqdm.auto import tqdm

## Scope settings

Choose `scope_kind = "visa"` (default), `"cw"` for a ChipWhisperer Lite/Husky, or `"pico"` for a PicoScope 3000E.
Voltage, sampling rate, duration, and delay settings accept unit strings, such as `"2mV"`, `"1GS/s"`, `"1us"`, and `"-100ns"`.
Trace length also accepts sample counts (`"3000samples"`), and delay accepts samples or a percentage of trace length (`"-10%"`).
A negative delay requests pretrigger samples, included in the trace length. Numeric values still mean volts, samples/s, or seconds.
Timeout remains a number in seconds. Values depend on the target, wiring, and instrument.

For ChipWhisperer, `cw_sampling_rate` specifies the requested ADC rate.
Husky uses an ADC multiplier of 1, and Lite uses `clkgen_x1` for simplicity.
Durations are converted using the actual ADC rate after clock lock. Trigger pin `tio4` is configured as an input.
The CW305 target uses its own clock; the capture clock is independent.

For VISA, replace `visa_address` with your instrument resource (list resources with `pyvisa-shell`).
Use integer channel numbers. See the [VISA API](../docs/visa_scope.md) for supported instruments.

For PicoScope, install `pypicosdk` and the native PicoSDK driver; see the [PicoScope guide](../docs/picoscope.md).
The notebook imports the PicoScope wrapper only when `scope_kind = "pico"`.
Use channel names `"A"` through `"D"`. Set trigger probe attenuation to match the probe (1 for a direct connection, 10 for a 10:1 probe).
Trace coupling defaults to `"AC"`; trigger coupling defaults to `"DC"`. Resolution is 8 or 10 bits and affects the available sampling rates.
The requested sampling rate is stored before trace configuration, which determines the actual rate.

For VISA and PicoScope, scales are in V/div and offsets are the range center voltage in V.
Trigger and trace channels must differ. An explicit trigger threshold accepts a voltage string; `None` selects `offset + scale`.
PicoScope scales must match a supported range divided by five. The example uses zero trace offset with AC coupling for PicoScope.

In [ ]:
scope_kind = "visa"  # "visa", "cw", or "pico"

# common settings for all scopes
trace_length = "1us"
capture_delay = "0ns"  # e.g. "-100ns", "-10%", or "-100samples"
capture_timeout = 2.0  # seconds

# Optional ChipWhisperer settings
cw_sampling_rate = "100MS/s"
cw_gain_db = 25
cw_trigger_mode = "rising_edge"

# Shared VISA/PicoScope settings: adapt to your instrument and wiring.
scope_sampling_rate = "1.25GS/s" if scope_kind == "pico" else "1GS/s"
trigger_channel = "C" if scope_kind == "pico" else 3
trace_channel = "A" if scope_kind == "pico" else 1
trigger_mode = TriggerMode.EDGE_RISE
trigger_scale = "1V"
trigger_offset = "0V"
trigger_threshold = None  # e.g. "2.4V"; None selects offset + scale
trace_scale = "2mV"
trace_offset = "0V" if scope_kind == "pico" else "-990mV"

# VISA settings: replace the address.
visa_address = "USB::REPLACE_WITH_YOUR_INSTRUMENT"

# PicoScope settings
pico_trigger_coupling = "DC"
pico_probe_scale = 10.0  # direct connection; use 10.0 for a 10:1 probe
pico_trace_coupling = "AC"
pico_resolution = 8

In [ ]:
duration, duration_unit = ScopeBase.decode_time(trace_length)
delay, delay_unit = ScopeBase.decode_time(capture_delay, allow_percent=True)
if duration <= 0 or capture_timeout <= 0:
    raise ValueError("Trace length and timeout must be positive")


def capture_window_samples(rate):
    count = duration * rate if duration_unit == "seconds" else duration
    start = delay * rate if delay_unit == "seconds" else delay
    if delay_unit == "percent":
        start = delay * count / 100
    samples = round(count)
    delay_samples = round(start)
    if samples < 1 or start <= -count or delay_samples <= -samples:
        raise ValueError("Trace length must span at least one sample and pretrigger must be smaller than trace length")
    return samples, delay_samples


if scope_kind == "cw":
    # ChipWhisperer scope configuration
    scope = cw.scope()
    is_husky = bool(getattr(scope, "_is_husky", False))
    requested_rate = ScopeBase.decode_sampling_rate(cw_sampling_rate)
    scope.adc.decimate = 1
    scope.clock.clkgen_src = "system"
    # Clocking system differs between ChipWhisperer Lite and Husky; set the ADC clock source accordingly.
    if is_husky:
        scope.clock.adc_mul = 1
        scope.clock.clkgen_freq = requested_rate
    else:
        scope.clock.adc_src = "clkgen_x1"
        scope.clock.clkgen_freq = requested_rate
    scope.clock.reset_dcms()
    if not scope.clock.adc_locked:
        raise RuntimeError("ADC clock is not locked; wait briefly and configure again")
    # calculate the number of samples and pretrigger samples based on the requested durations and the actual ADC sampling rate
    sampling_rate = float(scope.clock.adc_freq) / int(scope.adc.decimate)
    samples, delay_samples = capture_window_samples(sampling_rate)
    presamples = max(-delay_samples, 0)
    offset = max(delay_samples, 0)
    maximum = getattr(scope.adc, "max_samples", 131124 if is_husky else 24400)
    if not 1 <= samples <= maximum or not 0 <= presamples < samples:
        raise ValueError("Requested capture length/pretrigger exceeds the board's sample limits")
    # configure the scope with the calculated values
    scope.adc.samples = samples
    scope.adc.presamples = presamples
    scope.adc.offset = offset
    scope.adc.timeout = capture_timeout
    scope.gain.db = cw_gain_db
    # use the default trigger signal (tio4) and configure it as an input
    scope.io.tio4 = "high_z"
    if is_husky:
        scope.trigger.module = "basic"
    scope.trigger.triggers = "tio4"
    scope.adc.basic_mode = cw_trigger_mode
elif scope_kind in ("visa", "pico"):
    if trigger_channel == trace_channel:
        raise ValueError("Use different trigger and trace channels")
    trigger_options = {}
    trace_options = {}
    if scope_kind == "visa":
        scope = Oscilloscope(visa_address, timeout=capture_timeout)
    else:
        from cw_plugins.scopes.PicoScope import PicoScope3000E

        scope = PicoScope3000E()
        trigger_options = {"coupling": pico_trigger_coupling, "probe_scale": pico_probe_scale}
        trace_options = {"coupling": pico_trace_coupling, "resolution": pico_resolution}
    scope.set_sampling_rate(scope_sampling_rate)
    scope.config_trigger_channel(
        trigger_channel, trigger_scale, trigger_offset, trigger_mode,
        threshold=trigger_threshold, **trigger_options,
    )
    scope.config_trace_channel(
        trace_channel, trace_scale, trace_offset, trace_length,
        delay=capture_delay, **trace_options,
    )
    sampling_rate = scope.get_sampling_rate()
else:
    raise ValueError("scope_kind must be 'cw', 'visa', or 'pico'")

print(f"Sampling rate: {sampling_rate / 1e6:.6g} MSa/s")

## Connect the FPGA target

The default target is `CW305ShellExampleAES128BitRTL` with the `aist` AES core. The driver selects the bundled bitstream and HWH memory map when files are omitted. RTL implementations are `aist`, `google`, or `rsm`; select the one matching your design. For a custom design, add matching `bsfile="design.bit"` and `hwh_file="design.hwh"` to `target_options`. Connect the CW305 power measurement output and trigger to the scope channels configured above. See the [hardware guide](../docs/hardware.md) for board preparation.

For other implementations or boards, replace `target_class` and `target_options`:

```python
from cw_plugins.targets import (
    CW305ShellExampleAES128BitHLS, CW305VexRISCVAESExample,
    SakuraXShellExampleAES128BitRTL, SakuraXShellExampleAES128BitHLS,
    SakuraXVexRISCVAESExample, SakuraX, ESP32,
)
# CW305 HLS: CW305ShellExampleAES128BitHLS, implementation="naive" or "rsm".
# CW305 software AES: CW305VexRISCVAESExample, masked=False or True.
# SAKURA-X RTL: program Kintex-7 FPGA with the matching shell/AES design before instantiating the target.
# For Linux, auto-detecting the serial ports with no options is possible.
target_class = SakuraXShellExampleAES128BitRTL
target_options = {
    "data_port": "/dev/ttyUSB0",   # FT2232H Channel A
    "reset_port": "/dev/ttyUSB1",  # Channel B of the same board
    "implementation": "aist",
}
# SAKURA-X HLS: SakuraXShellExampleAES128BitHLS, same ports, implementation="naive" or "rsm".
# SAKURA-X software AES: SakuraXVexRISCVAESExample, same ports, masked=False or True.
```

SAKURA-X Shell requires separate data/reset ports. On Linux with the repository's udev rules and one connected board, omit both ports for automatic detection; add `serial_number` to select among multiple boards. Other platforms require both ports. The old Shell `serial_port` option is deprecated. Supply `hwh_file` for a custom memory map. Connection does not perform a hardware reset; call `target.hard_reset()` on a connected SAKURA-X Shell target if recovery is needed before capture.

Run exactly one target connection; disconnect before changing targets.

In [ ]:
target_class = CW305ShellExampleAES128BitRTL
target_options = {"implementation": "aist"}
# For a custom design:
# target_options.update(bsfile="design.bit", hwh_file="design.hwh")
target = cw.target(scope, target_class, **target_options)

## Create a project and prepare capture

The project stores waveforms, plaintexts, ciphertexts, and a fixed AES key. `overwrite=False` prevents accidental replacement; use a new path for each acquisition or explicitly enable overwriting. The `.cwp` file and associated data directory must be kept together.

`trace_count` counts successful captures. `max_attempts` bounds retries so a missing trigger cannot leave the notebook looping indefinitely. Integer waveforms preserve the acquisition format; use `as_int=False` for floating-point waveforms where supported.

In [ ]:
project_path = Path("project_data/code_capture.cwp")
trace_count = 100
as_int = True
if trace_count < 1:
    raise ValueError("Require 1 <= trace_count")
# ensure the project directory exists and create a new project without overwriting existing data
project_path.parent.mkdir(parents=True, exist_ok=True)

project = cw.create_project(str(project_path), overwrite=False)
ktp = cw.ktp.Basic()
key = ktp.next_key()
print("Key:", key)

## Capture and save

Run this cell once per new project. Failed captures (`None`) are retried with a new plaintext. The project is saved even if capture is interrupted or an error occurs, preserving successful traces. If the retry limit is reached, inspect wiring, trigger settings, and target configuration before creating another capture.

Progress updates do not redraw waveforms, avoiding preview overhead during acquisition.

In [ ]:
try:
    with tqdm(total=trace_count, initial=len(project.traces), desc="Capturing traces") as progress:
        while len(project.traces) < trace_count:
            text = ktp.next_text()
            trace = cw.capture_trace(scope, target, text, key, as_int=as_int)
            if trace is None:
                raise RuntimeError("Trace capture failed; check the scope and target configuration")
            else:
                project.traces.append(trace)
            progress.update(1)

finally:
    project.save()
    print(f"Saved {len(project.traces)} traces to {project_path}")

## Optional waveform preview

Plot the last successful trace after acquisition. The horizontal axis uses the actual sampling rate and configured delay; integer amplitudes are ADC codes.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if len(project.traces):
    wave = project.traces[-1].wave
    if scope_kind == "pico":
        # PicoScope's time axis includes sample rounding and pretrigger/trim.
        axis_units = {"s": 1e6, "ms": 1e3, "us": 1, "ns": 1e-3, "ps": 1e-6}
        time_us = np.asarray(scope.get_last_time_axis()) * axis_units[scope.time_unit]
    else:
        if scope_kind == "cw":
            start_samples = scope.adc.offset - scope.adc.presamples
        else:
            _, start_samples = capture_window_samples(sampling_rate)
        time_us = (np.arange(len(wave)) + start_samples) / sampling_rate * 1e6
    amplitude_label = "ADC code" if as_int else "Amplitude"
    if scope_kind == "pico" and not as_int:
        amplitude_label = f"Amplitude ({'mV' if scope.output_unit == 'mv' else 'V'})"
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(time_us, wave)
    ax.set(xlabel="Time relative to trigger (us)", ylabel=amplitude_label)
    ax.grid(True)
    plt.show()

## Disconnect

After saving/previewing, release the project and hardware connections. Run this cell also after a capture error before reconnecting.

In [ ]:
try:
    project.close()
finally:
    try:
        target.dis()
    finally:
        if scope_kind == "visa":
            scope.resource.close()
        elif scope_kind == "pico":
            scope.close()
        else:
            scope.dis()